<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Faults in machine or server metrics

**[Faults in machine or server metrics](https://learning.nextia-ai.com/courses/anomaly-detection/m10/faults-in-machine-or-server-metrics/)** · Anomaly Detection: Find What Is Unusual · Case study 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** watch 38 metrics of real servers for faults. You compare a per-metric robust z-score, Isolation Forest on short windows and a live change-point score. You set each threshold for the same page budget, score each detector by the faults it finds, how late, and the false pages a day, and then run the choice once on a new server.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in about 1 minute after the setup |
| **Needs** | An internet connection for the setup cells (about 55 MB). No account and no key. |
| **You should know** | The robust z-score, from [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/); Isolation Forest, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/); change points, from [Change points](https://learning.nextia-ai.com/courses/anomaly-detection/m05/change-points/); alert events and minutes to detection, from [Many metrics at once](https://learning.nextia-ai.com/courses/anomaly-detection/m05/many-metrics-at-once/). |
| **Data** | SMD (Server Machine Dataset), MIT licence: real, anonymous metrics of servers of a large internet company, one row per minute |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon, laptop CPU), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m10/faults-in-machine-or-server-metrics/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L02-faults-in-machine-or-server-metrics/faults-in-machine-or-server-metrics-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, ruptures 1.1.10. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 ruptures==1.1.10
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'ruptures']:
    print(p, version(p))

## Setup: get the data

Run the next cell. It downloads the metrics and the fault labels of three servers from the original source, the OmniAnomaly repository on GitHub, at a fixed version. It puts them in a new folder `server-metrics` and checks each file's checksum. That is 12 files, about 55 MB. It needs an internet connection and no account, and takes about 10 to 30 seconds.

You should see `Ready: 12 files, 55.5 MB.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `server-metrics` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** SMD (Server Machine Dataset) by NetManAIOps, from the OmniAnomaly repository, <https://github.com/NetManAIOps/OmniAnomaly>, licensed under the [MIT licence](https://github.com/NetManAIOps/OmniAnomaly/blob/master/ServerMachineDataset/LICENSE): "Copyright (c) 2021 NetManAIOps-SMD". We use three of its 28 machines and change nothing ([dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L02-faults-in-machine-or-server-metrics/dataset.md)).

In [ ]:
# Setup: download three servers of SMD into server-metrics/ and check each file. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "train-1-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/train/machine-1-1.txt",
        "ecebd2933ccdc1126e2abbfb538762dbde16410cebc77257a7b96e1e1fbcf753"
    ],
    "test-1-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test/machine-1-1.txt",
        "bdb801d8d52ce6b0c4a20c310edf31b5cee9bc82eb84fb38e7d4a19309b16ddb"
    ],
    "label-1-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test_label/machine-1-1.txt",
        "cddf33441ae480c6074e173c897d4715eb893f24a7078d3d3e1b79b200c7cd19"
    ],
    "interp-1-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/interpretation_label/machine-1-1.txt",
        "ea2b75dacb9cd140d81962203b18706e8e85d70c87dcced69746a37ba133fa48"
    ],
    "train-2-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/train/machine-2-1.txt",
        "d6eb13ff74a537cf33686319fffdd4cdcb394862570a3acd519b4c37af97cd2e"
    ],
    "test-2-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test/machine-2-1.txt",
        "2d103661799271be958227907950a1c76803356442d60a4070671cc0abbefb20"
    ],
    "label-2-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test_label/machine-2-1.txt",
        "416dd20a447f38b8d9394382293ed44f509c1c3f99c4aa2fbff179498626dbdc"
    ],
    "interp-2-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/interpretation_label/machine-2-1.txt",
        "fe4e3a67039d46045b9a3b18fd4dd6138f5c228447fcefb0c5f4beb1f46ede37"
    ],
    "train-3-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/train/machine-3-1.txt",
        "c1186ac085809c0590467dd42f4f12180012ebd994c1e615250dbed2a3588111"
    ],
    "test-3-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test/machine-3-1.txt",
        "b110b204ba4919250351eba7aa253d73ab54af0cf18e272b9b284de5fa7df88d"
    ],
    "label-3-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/test_label/machine-3-1.txt",
        "549348a9c63ed922e85157885bcb21289a8c9f5af94b50147a21cb7b928220ec"
    ],
    "interp-3-1.txt": [
        "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/7fb0e0acf89ea49908896bcc9f9e80fcfff6baf4/ServerMachineDataset/interpretation_label/machine-3-1.txt",
        "52f67a2be8538138b614850392a232b3f96c9eda987eba41aa2c31b534fd3e2f"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("server-metrics").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    if path.exists() and sha256(path) == expected:
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder server-metrics and run again.")

os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files, {sum(Path(n).stat().st_size for n in FILES) / 1e6:.1f} MB.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

SMD has 28 servers. Each server has **38 metrics**, one value per minute, for about five weeks. The metrics have no names, and the values are scaled from 0 to 1. The first half of each server (`train`) has **no labels**. The second half (`test`) has a label for every minute: 1 in a fault, 0 if not.

We use three servers, chosen by a rule before any detector ran: the first server of each group. Two are for the comparison. The third is the new input, used once at the end.

Run the next cell. It sets the case study's rules, loads the three servers and describes the two comparison servers.

> **Check.** machine-1-1 has 8 fault windows and machine-2-1 has 13: 21 faults in all.

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

DEV = ['1-1', '2-1']          # the comparison: the first machine of groups 1 and 2
NEW = '3-1'                 # the new input: the first machine of group 3, used once at the end
GAP = 30                    # the course's rule: alerts less than 30 minutes apart are one alert event
REPEAT = 30                 # an alert that stays on pages again after 30 minutes
FIT_SHARE = 0.8             # learn normal on the first 80% of the training half ...
BUDGET = 1.0                # ... and set each threshold there for 1 page a day per machine (assumption)
PAGE_MINUTES = 15           # an engineer's minutes for one page (assumption)
MIN_SCALE = 0.01            # the smallest spread of a metric: 1% of its 0-1 range

def load(folder, m):
    folder = Path(folder)
    read = lambda name: pd.read_csv(folder / name, header=None).to_numpy(float)  # noqa: E731
    train, test = read(f"train-{m}.txt"), read(f"test-{m}.txt")
    label = pd.read_csv(folder / f"label-{m}.txt", header=None)[0].to_numpy(int)
    return train, test, label

def windows_of(label):
    """The labelled fault windows as (start, end) minute numbers, end included."""
    d = np.diff(np.r_[0, label, 0])
    return list(zip(np.where(d == 1)[0].tolist(), (np.where(d == -1)[0] - 1).tolist()))

data = {m: load(".", m) for m in DEV + [NEW]}
for m in DEV:
    train, test, label = data[m]
    w = windows_of(label)
    print(f"machine-{m}: train {train.shape}, test {test.shape} = {len(test) / 1440:.1f} days, "
          f"{label.mean():.1%} of test minutes in a fault, {len(w)} fault windows")
    print("   window lengths (minutes):", [b - a + 1 for a, b in w])

Next, the first checks: missing values, the range, and metrics that never move.

In [ ]:
train, test, label = data["1-1"]
print("missing values:", int(np.isnan(train).sum() + np.isnan(test).sum()))
print("smallest and largest value:", train.min(), test.min(), "|", train.max(), round(test.max(), 3))
print("metrics that never change in the training half:", int((train.std(axis=0) == 0).sum()), "of 38")
print("metrics that never change in the test half:", int((test.std(axis=0) == 0).sum()), "of 38")

There are no missing values, and every value is between 0 and 1. Some metrics never move in the training half. A spread of 0 would make a z-score divide by zero, so each spread is at least 1% of the range (`MIN_SCALE`).

Now one fault. Run the next cell. It draws metric 1 of machine-1-1 around the first labelled fault.

> **Check.** Metric 1 jumps from 0.086 to 0.495 as the fault starts.

In [ ]:
import matplotlib.pyplot as plt
a, b = windows_of(label)[0]
span = np.arange(a - 240, b + 61)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(span - a, test[span, 0], lw=0.8, color="black", label="metric 1")
ax.axvspan(0, b - a, color="orange", alpha=0.25, label="labelled fault")
ax.set_xlabel("minutes from the start of the fault"); ax.set_ylabel("value (0 to 1)")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()
before, after = test[a - 1, 0], test[a + 1, 0]
print(f"metric 1: {before:.3f} the minute before the fault, {after:.3f} one minute after its start")

## 2. How we score a detector

An on-call engineer gets a **page** (a message to the phone) when an alert starts. If the alert stays on, the engineer is paged again every 30 minutes. A fault is **found** when the detector alerts inside its window. The **minutes to detection** run from the start of the window to the first alert inside it. A page outside every fault window is a **false page**.

Run the next cell. It defines the scoring and tries it on 120 made-up minutes: one fault from minute 50 to 59, and alerts at minutes 10, 52 to 54, 70 to 85, and 100.

> **Check.** You should see pages at minutes [10, 52, 82]: 1 true and 2 false. The fault is found 2 minutes after its start. The alert from minute 70 to 85 pages once, at minute 82, 30 minutes after the page at minute 52. The alert at minute 100 does not page: the last page was only 18 minutes before.

In [ ]:
def events_of(flags, gap=GAP):
    """Alert minutes less than `gap` minutes apart become one alert event: (first minute, last minute)."""
    idx = np.flatnonzero(flags)
    if len(idx) == 0:
        return []
    cut = np.flatnonzero(np.diff(idx) > gap)
    starts = np.r_[idx[0], idx[cut + 1]]
    ends = np.r_[idx[cut], idx[-1]]
    return list(zip(starts.tolist(), ends.tolist()))

def pages_of(flags, repeat=REPEAT):
    """The minutes at which the on-call engineer is paged: the first alert minute, and again at the next alert
    minute that is at least `repeat` minutes after the last page (an alert that stays on pages again)."""
    pages, last = [], -10**9
    for t in np.flatnonzero(flags).tolist():
        if t - last >= repeat:
            pages.append(t)
            last = t
    return pages

def score_windows(flags, windows, repeat=REPEAT):
    """One row of the comparison: pages, false pages, faults found and minutes to detection.
    A fault is found when an alert minute falls inside its window; the time to detection is from the start of
    the window to the first such minute. A page is true when it falls inside a fault window."""
    flags = np.asarray(flags, bool)
    pages = pages_of(flags, repeat)
    days = len(flags) / 1440
    inside = np.zeros(len(flags), bool)
    near = np.zeros(len(flags), bool)               # a fault window, 60 minutes before and after
    delays = []
    for a, b in windows:
        inside[a:b + 1] = True
        near[max(a - 60, 0):b + 61] = True
        hit = np.flatnonzero(flags[a:b + 1])
        delays.append(int(hit[0]) if len(hit) else None)
    true = sum(1 for t in pages if inside[t])
    false = len(pages) - true
    found = [d for d in delays if d is not None]
    return {"pages": len(pages), "false pages": false, "false pages per day": round(false / days, 2),
            "precision": round(true / len(pages), 3) if pages else 0.0,
            "faults found": len(found), "faults": len(windows), "recall": round(len(found) / len(windows), 3),
            "median minutes to detection": float(np.median(found)) if found else None,
            "minutes to detection": delays, "alert minutes": int(flags.sum()),
            "false pages near a fault": sum(1 for t in pages if near[t] and not inside[t]),
            "alert events (course rule)": len(events_of(flags))}

# A tiny example: 120 minutes, one fault from minute 50 to minute 59, alerts at the minutes below.
flags = np.zeros(120, bool)
flags[[10, 52, 53, 54, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 100]] = True
print("pages at minutes:", pages_of(flags))
row = score_windows(flags, [(50, 59)])
print({k: row[k] for k in ("pages", "false pages", "faults found", "minutes to detection")})
tiny_pages, tiny_row = pages_of(flags), row

Each detector gives a score per minute. A **threshold** turns the score into alerts. We choose every threshold the same way, with no label:

1. Learn normal on the first 80% of the training half.
2. On the last 20% (the **calibration** minutes, about 3 to 4 days), choose the lowest threshold that pages at most **1 time a day**. That is the page budget, an assumption.
3. Run on the test half, and score it with the labels.

## 3. The baseline: a robust z-score per metric

For each metric, the median and the MAD of the fit minutes give a robust z-score, as in Module 2. The detector's score is the **largest |z| of the 38 metrics**, minute by minute.

> **Check.** On machine-1-1, the robust z finds 8 of 8 faults. On machine-2-1, it finds 7 of 13.

In [ ]:
def threshold_for_budget(scores, budget=BUDGET, count="pages"):
    """The lowest threshold that pages at most `budget` times a day on these minutes.
    count="events": count alert events with the course's 30-minute rule instead (for the failure case)."""
    counter = pages_of if count == "pages" else events_of
    days = len(scores) / 1440
    cands = np.unique(np.quantile(scores, np.linspace(0.90, 1.0, 2001)))
    lo, hi = 0, len(cands) - 1
    while lo < hi:                                  # events fall as the threshold rises
        mid = (lo + hi) // 2
        if len(counter(scores > cands[mid])) / days <= budget:
            hi = mid
        else:
            lo = mid + 1
    return float(cands[lo])

def robust_scale(fit):
    """Each metric's median and spread (1.4826 x MAD, at least MIN_SCALE) on the fit minutes."""
    med = np.median(fit, axis=0)
    spread = np.maximum(1.4826 * np.median(np.abs(fit - med), axis=0), MIN_SCALE)
    return med, spread

def robust_z_score(x, med, spread):
    """The baseline: the largest |robust z| over the 38 metrics, minute by minute."""
    return np.abs((x - med) / spread).max(axis=1)

robust_rows = {}
for m in DEV:
    train, test, label = data[m]
    n_fit = int(len(train) * FIT_SHARE)
    med, spread = robust_scale(train[:n_fit])
    threshold = threshold_for_budget(robust_z_score(train[n_fit:], med, spread))
    flags = robust_z_score(test, med, spread) > threshold
    robust_rows[m] = score_windows(flags, windows_of(label))
    print(f"machine-{m}: threshold |z| > {threshold:.1f};",
          {k: robust_rows[m][k] for k in ("pages", "false pages", "faults found", "faults", "minutes to detection")})

## 4. Isolation Forest on short windows

For each minute, the features are the mean and the standard deviation of each metric over the last 10 minutes: 76 features. Isolation Forest learns the fit minutes, and its score is how fast a minute is cut free.

> **Check.** On machine-1-1, it finds 5 of 8 faults, all of them more than 200 minutes late. On machine-2-1, it finds 8 of 13.

In [ ]:
from sklearn.ensemble import IsolationForest

def window_features(x, w=10):
    """For each minute: the mean and the standard deviation of each metric over the last w minutes."""
    df = pd.DataFrame(x)
    r = df.rolling(w, min_periods=1)
    return np.hstack([r.mean().to_numpy(), r.std().fillna(0).to_numpy()])

def iforest_model(fit_feats, seed=0):
    return IsolationForest(n_estimators=200, max_samples=1024, random_state=seed, n_jobs=1).fit(fit_feats)

for m in DEV:
    train, test, label = data[m]
    n_fit = int(len(train) * FIT_SHARE)
    t0 = time.time()
    f_train, f_test = window_features(train), window_features(test)
    model = iforest_model(f_train[:n_fit])
    threshold = threshold_for_budget(-model.score_samples(f_train[n_fit:]))
    row = score_windows(-model.score_samples(f_test) > threshold, windows_of(label))
    print(f"machine-{m} ({time.time() - t0:.1f} s):",
          {k: row[k] for k in ("pages", "false pages", "faults found", "faults", "minutes to detection")})

## 5. A live change-point score

PELT and binary segmentation (Module 5) cut a **finished** series. A page must come now. So at every minute, we look only at the **last 60 minutes** and ask: where is the best single cut, and how much does it lower the squared error? A large drop means that the level moved inside the last hour. That drop is the score.

Run the next cell. It does this by hand on 12 minutes of one metric, with at least 3 minutes on each side of the cut.

> **Check.** The best cut is before minute 7. It lowers the squared error by 12.00.

In [ ]:
# Twelve minutes of one scaled metric: the level moves up after minute 6.
x = np.array([0.1, -0.2, 0.0, 0.2, -0.1, 0.0, 2.1, 1.9, 2.0, 2.2, 1.8, 2.0])
total = ((x - x.mean()) ** 2).sum()                          # the squared error with no cut
for cut in range(3, 10):                                       # at least 3 minutes on each side
    left, right = x[:cut], x[cut:]
    cost = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
    print(f"cut before minute {cut + 1:2d}: squared error {cost:6.2f}, lowered by {total - cost:6.2f}")
best_gain = max(total - ((x[:c] - x[:c].mean()) ** 2).sum() - ((x[c:] - x[c:].mean()) ** 2).sum() for c in range(3, 10))

`change_score` does the same at every minute, for all 38 metrics at once (scaled by their spread), with at least 5 minutes on each side. It uses running sums, so it is fast.

In [ ]:
def change_score(z, w=60, m=5):
    """A live change-point score: in the last w minutes, the best single cut with at least m minutes on each
    side, and how much it lowers the squared error (the l2 cost of the ruptures package). z: scaled metrics."""
    n, d = z.shape
    S = np.vstack([np.zeros(d), np.cumsum(z, axis=0)])
    out = np.zeros(n)
    t = np.arange(w - 1, n)                         # the window is a = t - w + 1 ... t
    a = t - w + 1
    best = np.zeros(len(t))
    for k in range(m, w - m + 1):                   # the cut: k minutes before it, w - k after it
        c = a + k
        left = (S[c] - S[a]) / k
        right = (S[t + 1] - S[c]) / (w - k)
        gain = (k * (w - k) / w) * ((left - right) ** 2).sum(axis=1)
        best = np.maximum(best, gain)
    out[w - 1:] = best
    return out

for m in DEV:
    train, test, label = data[m]
    n_fit = int(len(train) * FIT_SHARE)
    med, spread = robust_scale(train[:n_fit])
    z_train, z_test = (train - med) / spread, (test - med) / spread
    threshold = threshold_for_budget(change_score(z_train)[n_fit:])
    row = score_windows(change_score(z_test) > threshold, windows_of(label))
    print(f"machine-{m}:", {k: row[k] for k in ("pages", "false pages", "faults found", "faults", "minutes to detection")})

Is it the same cut as the `ruptures` package? Run the next cell. It compares one minute with ruptures' binary segmentation with one cut and the `l2` cost.

> **Check.** The two numbers are the same.

In [ ]:
import ruptures as rpt
train, test, label = data["2-1"]
n_fit = int(len(train) * FIT_SHARE)
med, spread = robust_scale(train[:n_fit])
z = (test - med) / spread
ours = change_score(z)
t = 9800                                        # one minute: the last 60 minutes end here
seg = z[t - 59:t + 1]
algo = rpt.Binseg(model="l2", min_size=5, jump=1).fit(seg)
cut = algo.predict(n_bkps=1)
print("ruptures: cut after minute", cut[0], "of 60; squared error lowered by",
      round(algo.cost.error(0, 60) - algo.cost.sum_of_costs(cut), 2))
print("change_score at that minute:", round(ours[t], 2))

## 6. The fair comparison

> **Predict.** At a budget of 1 page a day per machine, which detector finds the most faults within 30 minutes: the simple robust z, Isolation Forest, or the change-point score? Write your answer down.

Run the next cell. It scores the three detectors on both comparison machines with the same rules and adds them up.

> **Check.** The change-point score finds 17 of 21 faults within 30 minutes, the robust z 13, and Isolation Forest 6.

In [ ]:
def detector_scores(train, test):
    """Fit the three detectors on the first part of the training half. Returns, for each detector, its scores on
    the calibration minutes (the rest of the training half), on the test half, and the seconds it took."""
    n_fit = int(len(train) * FIT_SHARE)
    fit, cal = train[:n_fit], train[n_fit:]
    med, spread = robust_scale(fit)
    out = {}
    t0 = time.time()
    out["robust z"] = (robust_z_score(cal, med, spread), robust_z_score(test, med, spread), time.time() - t0)
    t0 = time.time()
    f_tr, f_te = window_features(train), window_features(test)
    model = iforest_model(f_tr[:n_fit])
    out["Isolation Forest"] = (-model.score_samples(f_tr[n_fit:]), -model.score_samples(f_te), time.time() - t0)
    t0 = time.time()
    z_tr, z_te = (train - med) / spread, (test - med) / spread
    out["change point"] = (change_score(z_tr)[n_fit:], change_score(z_te), time.time() - t0)
    return out

def pooled(rows, days):
    """Add up the rows of several machines."""
    pg = sum(r["pages"] for r in rows)
    fa = sum(r["false pages"] for r in rows)
    found = sum(r["faults found"] for r in rows)
    nw = sum(r["faults"] for r in rows)
    delays = [d for r in rows for d in r["minutes to detection"] if d is not None]
    return {"pages": pg, "false pages": fa, "false pages per day": round(fa / days, 2),
            "precision": round((pg - fa) / pg, 3) if pg else 0.0,
            "faults found": found, "faults": nw, "recall": round(found / nw, 3),
            "found within 30 min": sum(1 for d in delays if d <= 30),
            "median minutes to detection": float(np.median(delays)) if delays else None,
            "engineer hours a week on false pages": round(fa / days * 7 * PAGE_MINUTES / 60, 1)}

scores = {m: detector_scores(*data[m][:2]) for m in DEV}
dev_days = sum(len(data[m][1]) for m in DEV) / 1440

def compare(budget):
    """The three detectors on the two comparison machines, at one budget, added up."""
    table = {}
    for name in scores[DEV[0]]:
        rows = []
        for m in DEV:
            cal, st, _ = scores[m][name]
            rows.append(score_windows(st > threshold_for_budget(cal, budget), windows_of(data[m][2])))
        table[name] = pooled(rows, dev_days)
    return pd.DataFrame.from_dict(table, orient="index")

comparison = compare(BUDGET)
print(comparison[["pages", "false pages per day", "precision", "faults found", "found within 30 min",
                  "median minutes to detection", "engineer hours a week on false pages"]].to_string())

The budget was set on the calibration minutes, but the test halves give more false pages: 2.68 to 3.81 a day, not 1. The calibration days were calmer than the test days. So compare the detectors at the **same** false pages a day too. The next cell repeats the comparison for ten budgets and draws one line per detector.

In [ ]:
budgets = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
curves = {b: compare(b) for b in budgets}
fig, ax = plt.subplots(figsize=(7, 4))
for name in comparison.index:
    pts = [(curves[b].loc[name, "false pages per day"], curves[b].loc[name, "found within 30 min"]) for b in budgets]
    ax.plot(*zip(*pts), marker="o", label=name)
ax.set_xlabel("false pages a day per machine (test halves)"); ax.set_ylabel("faults found within 30 minutes (of 21)")
ax.legend(); plt.tight_layout(); plt.show()

Up to about 7 false pages a day, the change-point line is above the others. The robust z is close behind it, and catches up above about 7. Isolation Forest is far behind until about 8 false pages a day.

## 7. Explain a page

A page must say **which metric** to look at. For the chosen detector's first alert in each found fault, we name the metric with the largest |robust z|, and check it against the dataset's own list of the metrics behind each fault (`interp-*.txt`, metric numbers from 1).

In [ ]:
def read_interp(folder, m):
    """interp-<m>.txt: lines "start-end:d1,d2,..." (the dimensions that the dataset's authors marked)."""
    rows = []
    for line in (Path(folder) / f"interp-{m}.txt").read_text().split():
        span, dims = line.split(":")
        a, b = span.split("-")
        rows.append(((int(a), int(b)), [int(d) for d in dims.split(",")]))
    return rows

def explain(test, med, spread, flags, windows, interp):
    """For each found window: the metric with the largest |robust z| at the first alert minute inside the
    window, and whether the dataset's interpretation label lists it (metric numbers from 1)."""
    out = []
    for (a, b), dims in zip(windows, interp):
        hit = np.flatnonzero(flags[a:b + 1])
        if len(hit) == 0:
            out.append(None)
            continue
        t = a + hit[0]
        top = int(np.argmax(np.abs((test[t] - med) / spread))) + 1
        out.append({"window": [a, b], "minute": int(t), "top metric": top, "in label": top in dims})
    return out

CHOSEN = "change point"
for m in DEV:
    train, test, label = data[m]
    n_fit = int(len(train) * FIT_SHARE)
    med, spread = robust_scale(train[:n_fit])
    cal, st, _ = scores[m][CHOSEN]
    flags = st > threshold_for_budget(cal, BUDGET)
    interp = read_interp(".", m)
    found = [e for e in explain(test, med, spread, flags, [w for w, _ in interp], [d for _, d in interp]) if e]
    print(f"machine-{m}: the top metric is in the dataset's label for {sum(e['in label'] for e in found)} "
          f"of {len(found)} found faults")
    print("   ", [(e["top metric"], e["in label"]) for e in found])

## 8. The new server, once

The choice is made: the change-point score, explained by the metric with the largest |z|. Now we run all three detectors, with the same rules and no change, on machine-3-1. No step has seen it.

> **Predict.** Will the chosen detector keep its lead on a server from another group?

> **Check.** The change-point score finds 4 of 4 faults with 2.31 false pages a day.

In [ ]:
train, test, label = data[NEW]
new_scores = detector_scores(train, test)
new_rows = {}
for name, (cal, st, secs) in new_scores.items():
    new_rows[name] = score_windows(st > threshold_for_budget(cal, BUDGET), windows_of(label))
print(f"machine-{NEW}: {len(test) / 1440:.1f} test days, {len(windows_of(label))} fault windows")
print(pd.DataFrame.from_dict(new_rows, orient="index")[["pages", "false pages per day", "faults found", "minutes to detection",
                                "alert minutes"]].to_string())

The change-point score finds all 4 faults, 6 to 21 minutes after their start. The robust z finds only 2. Isolation Forest finds all 4, but with **24.18 false pages a day**: its normal moved between the training and the test half.

### A common mistake: counting alert events for the budget

The course counted alerts less than 30 minutes apart as one **alert event**. What happens if the page budget counts those events? Run the next cell: the robust z on machine-3-1, with the budget in alert events.

In [ ]:
cal, st, _ = new_scores["robust z"]
flags = st > threshold_for_budget(cal, BUDGET, count="events")   # the budget counts alert events, not pages
events = events_of(flags)
print("alert events:", len(events), "| alert minutes:", int(flags.sum()), "| pages:", len(pages_of(flags)))
print("the longest alert event:", max(e - s + 1 for s, e in events), "minutes")

The detector keeps its budget of events, but one event lasts **14,967 minutes**, more than 10 days. The metric alerts every few minutes, so the gaps never reach 30 minutes, and the alerts never stop. An engineer would get 928 pages, or **mute** the alert. Count pages, or cap the length of an event.

## 9. Check yourself

**Your turn 1.** The change-point score needs at least `m=5` minutes on each side of the cut. Run it with `m=10` on the two comparison machines, with the same rules, at the budget. How many faults does it find within 30 minutes? Write the number in `found_30_m10`. (Use `change_score(z, m=10)`, `threshold_for_budget`, `score_windows` and `pooled`.)

In [ ]:
rows = []
for m in DEV:
    train, test, label = data[m]
    n_fit = int(len(train) * FIT_SHARE)
    med, spread = robust_scale(train[:n_fit])
    z_train, z_test = (train - med) / spread, (test - med) / spread
    threshold = threshold_for_budget(change_score(z_train, m=10)[n_fit:])
    rows.append(score_windows(change_score(z_test, m=10) > threshold, windows_of(label)))
found_30_m10 = pooled(rows, dev_days)["found within 30 min"]
print(found_30_m10)

In [ ]:
expect('found_30_m10', found_30_m10, 16)

**Your turn 2.** Run the comparison at a budget of 2 pages a day. How many false pages a day does the robust z give on the comparison machines? Write it in `robust_false_per_day_2`. (Use `compare`.)

In [ ]:
robust_false_per_day_2 = compare(2.0).loc["robust z", "false pages per day"]
print(robust_false_per_day_2)

In [ ]:
expect('robust_false_per_day_2', robust_false_per_day_2, 3.62, tolerance=0.001)

## Recap

- On 38 real server metrics, we set every threshold the same way, with no label: 1 page a day on calibration minutes.
- The live change-point score found the most faults within 30 minutes (17 of 21), with more false pages than the robust z. The simple robust z was close behind (13). Isolation Forest on windows was late and missed more (6).
- The budget did not hold on the test days. Compare the detectors at the same false pages a day too.
- On a new server, the choice found all 4 faults. Isolation Forest flooded: 24.18 false pages a day.
- Count pages for a budget, not alert events: one event can last 10 days.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Unusual readings in public open data](https://learning.nextia-ai.com/courses/anomaly-detection/m10/unusual-readings-in-public-open-data/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m10/faults-in-machine-or-server-metrics/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). SMD is real, anonymous server data under the MIT licence; the people in the lesson are invented.